# 05 -- Response spectra (Newmark, RotD)

In [1]:
%matplotlib inline

import numpy as np
from asdea_sensors import SensorDataset
from asdea_sensors.config import settings

# The example data folder (edit to your path if needed).
DATA = r"C:\Users\ppala\Desktop\02_31MAY2026"

# A short window keeps every example fast (one ~10 min file).
WSTART, WLEN = "2026-05-31 15:00:00", "10min"

In [2]:
ds = SensorDataset(DATA, verbose=True)
ds.devices

------------------------------------------------------------
SensorDataset
------------------------------------------------------------
path        : C:\Users\ppala\Desktop\02_31MAY2026
files       : 32
time span   : 2026-05-31 14:52:12  ->  2026-05-31 20:02:13
duration    : 18601.0 s
devices     : MNAT0031, MNAT0034, MOF00134, MOF00135, MOF00136
fs / dt     : 252.5885 Hz / 0.003959 s
------------------------------------------------------------
axes (per sensor):
  MNAT0031   -> (3, 1, 5)
  MNAT0034   -> (3, 1, 5)
  MOF00134   -> (0, 1, 2)
  MOF00135   -> (3, 1, 5)
  MOF00136   -> (3, 1, 5)
------------------------------------------------------------
on-disk size: 782.17 MB
RAM         : used 33.15 GB / avail 0.36 GB (99%)
------------------------------------------------------------


['MNAT0031', 'MNAT0034', 'MOF00134', 'MOF00135', 'MOF00136']

In [3]:
wf = ds.MOF00135.window(WSTART, WLEN).baseline().filter(0.1, 24.9)

## Newmark, full output

In [4]:
spec = wf.newmark(component="x", zeta=0.05, max_period=5.01, dT=0.01, factor=1.0)
for k in ["T","PSa","PSv","Sd","Sv","Sa","u","v","a","at"]:
    print(k, np.shape(spec[k]))

C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\src\asdea_sensors\core\h5_reader.py:45: UserWarning: H5Reader: axes (3, 1, 5) for 'MOF00135' exceed the 3 available columns; falling back to (0, 1, 2)
  warnings.warn(


C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\src\asdea_sensors\derive\filters.py:49: UserWarning: obspy not available; falling back to the scipy engine
  warnings.warn("obspy not available; falling back to the scipy engine")


[newmark] MOF00135 comp=x zeta=0.05 Tmax=5.01 dT=0.01 -> 501 periods (computed)
T (501,)
PSa (501,)
PSv (501,)
Sd (501,)
Sv (501,)
Sa (501,)
u (149240,)
v (149240,)
a (149240,)
at (149240,)


## Present PSa in g

In [5]:
spec_g = wf.newmark(component="x", factor=1/9.81)
print("PSa max [m/s^2]:", round(float(spec["PSa"].max()),5), " [g]:", round(float(spec_g["PSa"].max()),6))

[newmark] MOF00135 comp=x zeta=0.05 Tmax=5.01 dT=0.01 -> 501 periods (computed)
PSa max [m/s^2]: 0.00938  [g]: 0.000956


## All three components

In [6]:
specs = wf.newmark(component="all")
print("axes:", list(specs.keys()))

[newmark] MOF00135 comp=x zeta=0.05 Tmax=5.01 dT=0.01 -> 501 periods (cached)


[newmark] MOF00135 comp=y zeta=0.05 Tmax=5.01 dT=0.01 -> 501 periods (computed)


[newmark] MOF00135 comp=z zeta=0.05 Tmax=5.01 dT=0.01 -> 501 periods (computed)
axes: ['x', 'y', 'z']


## RotD percentiles (coarse for speed; matrix cached)

In [7]:
r50  = wf.rotd(comp_x="x", comp_y="y", rotd=50, angle_step=15, max_period=3.0, dT=0.02)
r100 = wf.rotd(rotd=100, angle_step=15, max_period=3.0, dT=0.02)
print("ROTD50 max:", round(float(r50["ROTD50"].max()),5), " ROTD100 max:", round(float(r100["ROTD100"].max()),5))

[rotd] MOF00135 x/y rotd=50 -> 150 periods (computed)


[rotd] MOF00135 x/y rotd=100 -> 150 periods (computed)
ROTD50 max: 0.00871  ROTD100 max: 0.00949
